# ============================================================
# Author: Mayur Deshmukh
# Title: 02_eda.ipynb
# Project: ML-Binary-Classifier-For-Stock-Price-Prediction
# Purpose: Exploratory Data Analysis
# Python Version: 3.11
# ============================================================

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')
output_dir = os.path.join('..', '..', 'output')

In [ ]:
# ── Load Dataset ────────────────────────────────────────────────────────────────

df = pd.read_csv(os.path.join(output_dir, 'filtered_stocks.csv'), parse_dates=['date'])
print(f"Shape: {df.shape}")
print(f"Tickers: {df['Name'].unique().tolist()}")
df.head()

In [ ]:
# ── Descriptive Statistics ──────────────────────────────────────────────────────

print("=== Data Types & Nulls ===")
print(df.info())

print("\n=== Descriptive Statistics (numeric) ===")
display(df.describe().T.round(2))

print("\n=== Target Distribution (overall) ===")
target_dist = df['target'].value_counts(normalize=True).mul(100).round(2).rename({0: 'Down %', 1: 'Up %'})
print(target_dist)

print("\n=== Target Distribution per Ticker ===")
ticker_target = (df.groupby('Name')['target']
                   .value_counts(normalize=True)
                   .mul(100).round(2)
                   .rename('pct').reset_index())
ticker_target['target'] = ticker_target['target'].map({1: 'Up', 0: 'Down'})
display(ticker_target.pivot(index='Name', columns='target', values='pct').reset_index())

In [ ]:
# ── Univariate Analysis ─────────────────────────────────────────────────────────

num_cols = ['open', 'high', 'low', 'close', 'volume']

fig, axes = plt.subplots(2, len(num_cols), figsize=(20, 8))
for i, col in enumerate(num_cols):
    # Histogram
    axes[0, i].hist(df[col], bins=50, edgecolor='k', alpha=0.7)
    axes[0, i].set_title(f'{col} — Distribution')
    axes[0, i].set_xlabel(col)
    # Boxplot
    axes[1, i].boxplot(df[col], vert=True, patch_artist=True)
    axes[1, i].set_title(f'{col} — Boxplot')

plt.suptitle('Univariate Analysis: OHLCV Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'univariate_analysis.png'), bbox_inches='tight')
plt.show()

# Target class balance bar chart
fig, ax = plt.subplots(figsize=(5, 4))
df['target'].value_counts().rename({0: 'Down (0)', 1: 'Up (1)'}).plot(kind='bar', ax=ax, color=['salmon', 'steelblue'], edgecolor='k')
ax.set_title('Target Class Distribution')
ax.set_ylabel('Count')
ax.set_xlabel('Target')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'target_distribution.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Bivariate Analysis ──────────────────────────────────────────────────────────

# Boxplots of OHLCV features split by target
fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 5))
for i, col in enumerate(num_cols):
    df.boxplot(column=col, by='target', ax=axes[i], patch_artist=True)
    axes[i].set_title(f'{col} by Target')
    axes[i].set_xlabel('Target (0=Down, 1=Up)')

plt.suptitle('Bivariate Analysis: Features vs Target', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'bivariate_analysis.png'), bbox_inches='tight')
plt.show()

# Mean feature values per target
print("=== Mean feature values by target ===")
display(df.groupby('target')[num_cols].mean().round(2))

# Close price distribution per ticker coloured by target
fig, axes = plt.subplots(1, len(df['Name'].unique()), figsize=(20, 4), sharey=False)
for ax, ticker in zip(axes, sorted(df['Name'].unique())):
    sub = df[df['Name'] == ticker]
    for label, color in [(0, 'salmon'), (1, 'steelblue')]:
        sub[sub['target'] == label]['close'].plot(kind='hist', bins=40, alpha=0.6, color=color, ax=ax)
    ax.set_title(ticker)
    ax.set_xlabel('Close Price')
axes[0].legend(['Down', 'Up'])
plt.suptitle('Close Price Distribution by Target per Ticker', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'close_by_target_per_ticker.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Analysis ────────────────────────────────────────────────────────

corr_cols = num_cols + ['target']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (OHLCV + Target)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'correlation_matrix.png'), bbox_inches='tight')
plt.show()

print("=== Correlation with Target ===")
print(corr_matrix['target'].drop('target').sort_values(ascending=False).round(4))

In [ ]:
# ── Time Series: Closing Price per Ticker ───────────────────────────────────────

fig, axes = plt.subplots(len(df['Name'].unique()), 1,
                         figsize=(16, 4 * len(df['Name'].unique())), sharex=False)

for ax, ticker in zip(axes, sorted(df['Name'].unique())):
    sub = df[df['Name'] == ticker].set_index('date')
    ax.plot(sub['close'], label='Close', linewidth=0.8)
    up_days   = sub[sub['target'] == 1]
    down_days = sub[sub['target'] == 0]
    ax.scatter(up_days.index,   up_days['close'],   color='steelblue', s=3, alpha=0.5, label='Up')
    ax.scatter(down_days.index, down_days['close'], color='salmon',    s=3, alpha=0.5, label='Down')
    ax.set_title(f'{ticker} — Close Price with Target Labels')
    ax.set_ylabel('Close')
    ax.legend(loc='upper left', markerscale=4, fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'close_price_timeseries.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ── Save EDA-ready Dataset ──────────────────────────────────────────────────────

eda_path = os.path.join(output_dir, 'eda_stocks.csv')
df.to_csv(eda_path, index=False)
print(f"EDA dataset saved to: {eda_path} | Shape: {df.shape}")